<a href="https://colab.research.google.com/github/GH-Rumi/PemMes_7_Helmi/blob/main/JS06/JS06_Tugas1_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import cv2
import os
import glob
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder, StandardScaler

In [9]:
import zipfile

zip_path = 'images.zip'
extract_path = './'
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

def ekstrak_histogram(image_path):
    image = cv2.imread(image_path)
    if image is None: return None
    image_hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    hist = cv2.calcHist([image_hsv], [2], None, [256], [0, 256])
    cv2.normalize(hist, hist)
    return hist.flatten()

data_citra = []
label_citra = []
base_dir = 'images'
kategori = ['day', 'night']

for sub_dir in ['training', 'test']:
    for label in kategori:
        folder_path = os.path.join(base_dir, sub_dir, label)
        if os.path.exists(folder_path):
            for img_file in glob.glob(os.path.join(folder_path, '*.jpg')):
                hist_fitur = ekstrak_histogram(img_file)
                if hist_fitur is not None:
                    data_citra.append(hist_fitur)
                    label_citra.append(1 if label == 'day' else 0)

X_img = np.array(data_citra)
y_img = np.array(label_citra)

In [10]:
if len(X_img) > 0:
    X_train_img, X_test_img, y_train_img, y_test_img = train_test_split(
        X_img, y_img, test_size=0.2, random_state=42
    )

    param_grid = {
        'C': [0.1, 1, 10, 100],
        'gamma': ['scale', 'auto', 0.1, 0.01],
        'kernel': ['rbf']
    }

    grid_search = GridSearchCV(SVC(), param_grid, refit=True, verbose=1, cv=5)
    grid_search.fit(X_train_img, y_train_img)

    print("\\nParameter RBF terbaik ditemukan:", grid_search.best_params_)

    y_pred_img = grid_search.predict(X_test_img)
    akurasi_img = accuracy_score(y_test_img, y_pred_img)

    print(f"Akurasi Model Klasifikasi Siang/Malam (Split 80:20): {akurasi_img * 100:.2f}%")

Fitting 5 folds for each of 16 candidates, totalling 80 fits
\nParameter RBF terbaik ditemukan: {'C': 100, 'gamma': 'scale', 'kernel': 'rbf'}
Akurasi Model Klasifikasi Siang/Malam (Split 80:20): 98.75%
